In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import classification_report, brier_score_loss

In [ ]:
df = pd.read_excel('data/Swan Consulting 1 - Project Data.xlsx', sheet_name='Telco_Churn')

In [ ]:
df_clean = df.copy()
df_clean = df_clean.drop(columns=['Count', 'State', 'Country'])
df_clean = df_clean.drop(columns= ['Lat Long', 'City','Zip Code'])
df_clean = df_clean.drop(columns= ['Churn Label'])
df_clean = df_clean.drop(columns= ['Churn Reason'])
df_clean = df_clean.drop(columns= ['Latitude', 'Longitude'])

In [5]:
df_clean['Gender'] = df_clean['Gender'].map({'Male':0, 'Female':1})
df_clean['Senior Citizen'] = df_clean['Senior Citizen'].map({'No':0, 'Yes':1})
df_clean['Partner'] = df_clean['Partner'].map({'No':0, 'Yes':1})
df_clean['Dependents'] = df_clean['Dependents'].map({'No':0, 'Yes':1})
df_clean['Phone Service'] = df_clean['Phone Service'].map({'No':0, 'Yes':1})
df_clean['Paperless Billing'] = df_clean['Paperless Billing'].map({'No':0, 'Yes':1})

In [6]:
df_clean['Multiple Lines'] = df_clean['Multiple Lines'].replace('No phone service', 'No')

service_cols = ['Online Security', 'Online Backup', 'Device Protection',
                 'Tech Support', 'Streaming TV', 'Streaming Movies']
for col in service_cols:
    df_clean[col] = df_clean[col].replace('No internet service', 'No')

df_clean = pd.get_dummies(df_clean, columns=['Multiple Lines'], drop_first=True)
df_clean = pd.get_dummies(df_clean, columns=[
    'Internet Service',
    'Online Security',
    'Online Backup',
    'Device Protection',
    'Tech Support',
    'Streaming TV',
    'Streaming Movies',
    'Contract',
    'Payment Method'
], drop_first=True)

In [7]:
df_clean = df_clean.astype({col: int for col in df_clean.select_dtypes(include='bool').columns})
df_clean[pd.to_numeric(df_clean['Total Charges'], errors='coerce').isna()][['Total Charges', 'Tenure Months']]
df_clean['Total Charges'] = pd.to_numeric(df_clean['Total Charges'], errors='coerce').fillna(0)

In [ ]:
df_clean['log_tenure'] = np.log1p(df_clean['Tenure Months'])
df_clean['tenure_sq'] = df_clean['Tenure Months'] ** 2
df_clean['solo_no_deps'] = ((df_clean['Partner'] == 0) & (df_clean['Dependents'] == 0)).astype(int)
df_clean['new_customer'] = (df_clean['Tenure Months'] < 12).astype(int)

In [ ]:
X = df_clean.drop(columns=['Churn Value', 'CustomerID'])
y = df_clean['Churn Value']

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def compute_vif(X):
    X_const = X.assign(const=1)
    return pd.Series(
        [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1] - 1)],
        index=X.columns
    )

features = X.columns.tolist()
while True:
    vif = compute_vif(X[features])
    max_vif = vif.max()
    if max_vif <= 10:
        break
    drop_col = vif.idxmax()
    print(f"Dropping '{drop_col}' (VIF={max_vif:.1f})")
    features.remove(drop_col)

print("\nFinal VIF (all <= 10):")
print(vif.sort_values(ascending=False))

X = X[features]

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

param_grid = {
    'scaler': [StandardScaler(), RobustScaler()],
    'model__C': np.logspace(-3, 2, 20),
    'model__penalty': ['l1', 'l2'],
    'model__solver': ['liblinear', 'saga']
}

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(pipeline, param_grid, cv=kfold, scoring='roc_auc')
grid_search.fit(X, y)

print(grid_search.best_params_)

In [ ]:
oof_proba = cross_val_predict(grid_search.best_estimator_, X, y, cv=kfold, method='predict_proba')[:, 1]

In [ ]:
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
cv_results = cross_validate(grid_search.best_estimator_, X, y, cv=kfold, scoring=scoring)

for metric in scoring:
    scores = cv_results[f'test_{metric}']
    print(f"{metric}: {scores.mean():.3f} +/- {scores.std():.3f}  {scores.round(3)}")

brier = brier_score_loss(y, oof_proba)
print(f"brier: {brier:.3f}")

In [ ]:
oof_pred = (oof_proba >= 0.5).astype(int)
print(classification_report(y, oof_pred))

In [13]:
model = grid_search.best_estimator_.named_steps['model']
coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', ascending=False)

print(coefficients)

                                   Feature  Coefficient
8             Internet Service_Fiber optic     0.797589
19         Payment Method_Electronic check     0.394268
6                        Paperless Billing     0.323466
15                    Streaming Movies_Yes     0.270213
14                        Streaming TV_Yes     0.255823
7                       Multiple Lines_Yes     0.255760
2                                  Partner     0.198394
1                           Senior Citizen     0.060703
20             Payment Method_Mailed check     0.043839
12                   Device Protection_Yes    -0.030180
0                                   Gender    -0.031626
18  Payment Method_Credit card (automatic)    -0.042617
11                       Online Backup_Yes    -0.105128
13                        Tech Support_Yes    -0.317952
5                            Phone Service    -0.344363
10                     Online Security_Yes    -0.352944
16                       Contract_One year    -0

In [ ]:
pd.DataFrame({'CustomerID': df_clean['CustomerID'],
              'churn_probability': oof_proba.round(4)}).to_csv('churn_scores.csv', index=False)